# Registration Mask Comparison

This notebook evaluates candidate brain masks for constructing the 3D diffusion registration reference used by `bbregister`.

The reconstructed pipeline generates masks using FSL BET. Because BET output depends on the fractional intensity threshold and may differ from the legacy subject-space mask, candidate masks are compared while keeping the reconstructed diffusion image fixed.

The legacy mask is used only as a benchmark.

The comparison focuses on:

1. candidate mask inventory,
2. spatial coverage,
3. registration-reference similarity,
4. identification of the best independently generated mask.

Only existing preprocessing outputs are read. No preprocessing files are overwritten.

In [11]:
from pathlib import Path
import os
import subprocess

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

## 1. Configuration

All project paths and the subject ID are defined in one place so that this notebook can be reused for other subjects and projects.

In [12]:
# ============================================================
# USER CONFIGURATION
# ============================================================

SUBJECT = os.environ.get(
    "VALIDATION_SUBJECT",
    "D0007",
)

WORK_ROOT = Path(
    os.environ.get(
        "PAROS_WORK",
        "/mnt/newStor/paros/paros_WORK/xintong",
    )
)

CODE_ROOT = Path(
    os.environ.get(
        "TRAJECTORY_CODE_ROOT",
        WORK_ROOT / "code" / "cortical_trajectory_analysis",
    )
)

TEST_ROOT = Path(
    os.environ.get(
        "TRAJECTORY_TEST_ROOT",
        WORK_ROOT / "test",
    )
)

OLD_PIPELINE_ROOT = (
    TEST_ROOT
    / "existing_pipeline"
)

NEW_PIPELINE_ROOT = (
    TEST_ROOT
    / "new_pipeline"
    / SUBJECT
)

RESULTS_ROOT = (
    CODE_ROOT
    / "validation"
    / "results"
    / SUBJECT
    / "registration_mask_comparison"
)

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 70)
print("REGISTRATION MASK COMPARISON")
print("=" * 70)
print(f"Subject           : {SUBJECT}")
print(f"Old pipeline root : {OLD_PIPELINE_ROOT}")
print(f"New pipeline root : {NEW_PIPELINE_ROOT}")
print(f"Results root      : {RESULTS_ROOT}")

REGISTRATION MASK COMPARISON
Subject           : D0007
Old pipeline root : /mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline
New pipeline root : /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007
Results root      : /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison


## 2. Reusable File Discovery

Files are located recursively from configurable project roots rather than by hard-coded full paths.

The search functions return all matching candidates unless a unique file is explicitly required.

In [13]:
def find_files(
    root,
    patterns,
    subject=None,
):
    """
    Recursively search for files matching one or more patterns.
    """
    root = Path(root)

    if isinstance(patterns, str):
        patterns = [patterns]

    matches = []

    for pattern in patterns:
        matches.extend(
            root.rglob(pattern)
        )

    matches = sorted(
        set(matches)
    )

    if subject is not None:
        subject_matches = [
            p for p in matches
            if subject.lower()
            in str(p).lower()
        ]

        if subject_matches:
            matches = subject_matches

    return matches


def find_one(
    root,
    patterns,
    subject=None,
    required=True,
):
    """
    Return exactly one matching file.
    """
    matches = find_files(
        root=root,
        patterns=patterns,
        subject=subject,
    )

    if len(matches) == 0:

        if required:
            raise FileNotFoundError(
                f"No matching file found.\n"
                f"Root: {root}\n"
                f"Patterns: {patterns}\n"
                f"Subject: {subject}"
            )

        return None

    if len(matches) > 1:

        print(
            f"Found {len(matches)} candidates:"
        )

        for i, path in enumerate(matches):
            print(
                f"[{i}] {path}"
            )

        raise RuntimeError(
            "Multiple matching files found. "
            "Refine the search pattern."
        )

    return matches[0]

## 3. Locate Required Inputs

The comparison requires:

- the legacy 3D masked diffusion registration reference,
- the reconstructed bias-corrected 4D DWI,
- legacy mask candidates,
- reconstructed mask candidates.

No files are generated in this step.

In [14]:
# ============================================================
# LEGACY REGISTRATION REFERENCE
# ============================================================

OLD_REG_REFERENCE = find_one(
    OLD_PIPELINE_ROOT,
    patterns=[
        "*dwi_masked.nii.gz",
        "*DWI_masked.nii.gz",
    ],
    subject=SUBJECT,
)


# ============================================================
# RECONSTRUCTED BIAS-CORRECTED DWI
# ============================================================

BIAS_CORRECTED_DWI = find_one(
    NEW_PIPELINE_ROOT,
    patterns=[
        "*05*dwi_biascorrected*.mif",
        "*biascorrected*.mif",
    ],
    subject=SUBJECT,
)


# ============================================================
# MASK CANDIDATES
# ============================================================

def get_mask_candidates(root, subject):
    """
    Find mask images while excluding masked diffusion/reference images.
    """
    candidates = find_files(
        root,
        patterns=[
            "*mask*.nii.gz",
        ],
        subject=subject,
    )

    excluded_terms = [
        "dwi_masked",
        "sum_masked",
        "summed_masked",
    ]

    return [
        path for path in candidates
        if not any(
            term in path.name.lower()
            for term in excluded_terms
        )
    ]


OLD_MASK_CANDIDATES = get_mask_candidates(
    OLD_PIPELINE_ROOT,
    SUBJECT,
)

NEW_MASK_CANDIDATES = get_mask_candidates(
    NEW_PIPELINE_ROOT,
    SUBJECT,
)


print("Legacy registration reference:")
print(OLD_REG_REFERENCE)

print("\nBias-corrected DWI:")
print(BIAS_CORRECTED_DWI)

print("\nLegacy mask candidates:")
for i, path in enumerate(OLD_MASK_CANDIDATES):
    print(f"[{i}] {path}")

print("\nReconstructed mask candidates:")
for i, path in enumerate(NEW_MASK_CANDIDATES):
    print(f"[{i}] {path}")

Legacy registration reference:
/mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/D0007_dwi_masked.nii.gz

Bias-corrected DWI:
/mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_05_dwi_biascorrected.mif

Legacy mask candidates:

Reconstructed mask candidates:
[0] /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_b0_brain.nii_mask.nii.gz
[1] /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_eddy_mask.nii.gz
[2] /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_mean_b0_brain.nii_mask.nii.gz
[3] /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_mean_b0_brain_mask.nii.gz


## 4. Build the Common Summed Diffusion Image

To isolate the effect of mask choice, all candidate masks are applied to the same reconstructed diffusion image.

The 3D image is generated from the existing bias-corrected 4D DWI by:

1. excluding volume 0,
2. summing volumes 1:end.

The generated image is stored under the validation results directory and does not modify preprocessing outputs.

In [15]:
SUMMED_DWI = (
    RESULTS_ROOT
    / f"{SUBJECT}_biascorrected_dw_sum.nii.gz"
)

if SUMMED_DWI.exists():

    print(
        "Using existing validation output:"
    )
    print(
        SUMMED_DWI
    )

else:

    mrconvert_cmd = [
        "mrconvert",
        str(BIAS_CORRECTED_DWI),
        "-coord",
        "3",
        "1:end",
        "-",
    ]

    mrmath_cmd = [
        "mrmath",
        "-",
        "sum",
        str(SUMMED_DWI),
        "-axis",
        "3",
        "-force",
    ]

    p1 = subprocess.Popen(
        mrconvert_cmd,
        stdout=subprocess.PIPE,
    )

    subprocess.run(
        mrmath_cmd,
        stdin=p1.stdout,
        check=True,
    )

    p1.stdout.close()
    p1.wait()

    print(
        "Created validation-only summed DWI:"
    )
    print(
        SUMMED_DWI
    )

Using existing validation output:
/mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/D0007_biascorrected_dw_sum.nii.gz


## 5. Candidate Mask Inventory

Candidate masks are checked against the summed diffusion image.

Only masks with matching:

- image shape,
- affine geometry,

are retained for comparison.

The number of nonzero voxels is reported to characterize differences in spatial brain coverage.

In [16]:
summed_img = nib.load(
    SUMMED_DWI
)

summed_shape = (
    summed_img.shape
)

summed_affine = (
    summed_img.affine
)


def inspect_mask(path):

    img = nib.load(
        path
    )

    data = np.asarray(
        img.dataobj,
        dtype=np.float32,
    )

    return {
        "path": Path(path),
        "shape": data.shape,
        "affine": img.affine,
        "nonzero": int(
            np.count_nonzero(data)
        ),
        "min": float(
            np.nanmin(data)
        ),
        "max": float(
            np.nanmax(data)
        ),
    }


candidate_masks = []

all_mask_candidates = (
    OLD_MASK_CANDIDATES
    + NEW_MASK_CANDIDATES
)


for path in all_mask_candidates:

    try:
        info = inspect_mask(
            path
        )

    except Exception:
        continue

    same_shape = (
        info["shape"]
        == summed_shape
    )

    same_affine = np.allclose(
        info["affine"],
        summed_affine,
        atol=1e-5,
    )

    if (
        same_shape
        and same_affine
    ):

        source = (
            "OLD"
            if path
            in OLD_MASK_CANDIDATES
            else "NEW"
        )

        info["source"] = source

        candidate_masks.append(
            info
        )


mask_inventory_df = pd.DataFrame([
    {
        "Mask": x["path"].name,
        "Source": x["source"],
        "Nonzero voxels": x["nonzero"],
        "Min": x["min"],
        "Max": x["max"],
    }
    for x in candidate_masks
])

display(
    mask_inventory_df
)

,Mask,Source,Nonzero voxels,Min,Max
0,D0007_b0_brain.nii_mask.nii.gz,NEW,1435639,0.0,1.0
1,D0007_eddy_mask.nii.gz,NEW,1470842,0.0,1.0
2,D0007_mean_b0_brain.nii_mask.nii.gz,NEW,1487914,0.0,1.0
3,D0007_mean_b0_brain_mask.nii.gz,NEW,1470842,0.0,1.0


## 6. Registration Reference Comparison

Each compatible mask is applied to the same reconstructed summed diffusion image.

The resulting 3D registration reference is compared with the legacy registration reference using:

- spatial correlation,
- MAE,
- RMSE,
- nonzero voxel count.

This isolates the influence of mask selection while holding the reconstructed diffusion data constant.

In [17]:
old_ref_img = nib.load(
    OLD_REG_REFERENCE
)

old_ref = np.asarray(
    old_ref_img.dataobj,
    dtype=np.float32,
)


summed_data = np.asarray(
    summed_img.dataobj,
    dtype=np.float32,
)


mask_results = []


for info in candidate_masks:

    mask_path = (
        info["path"]
    )

    mask_img = nib.load(
        mask_path
    )

    mask_data = np.asarray(
        mask_img.dataobj,
        dtype=np.float32,
    )

    binary_mask = (
        mask_data > 0
    )

    candidate_ref = (
        summed_data
        * binary_mask
    )

    valid = (
        np.isfinite(old_ref)
        & np.isfinite(candidate_ref)
        & (
            (old_ref != 0)
            | (candidate_ref != 0)
        )
    )

    old_values = (
        old_ref[valid]
    )

    new_values = (
        candidate_ref[valid]
    )

    difference = (
        new_values
        - old_values
    )

    mask_results.append({
        "Mask": mask_path.name,
        "Source": info["source"],
        "Mask voxels": int(
            binary_mask.sum()
        ),
        "Reference voxels": int(
            np.count_nonzero(
                candidate_ref
            )
        ),
        "Correlation": np.corrcoef(
            old_values,
            new_values,
        )[0, 1],
        "MAE": np.mean(
            np.abs(
                difference
            )
        ),
        "RMSE": np.sqrt(
            np.mean(
                difference ** 2
            )
        ),
    })


mask_comparison_df = (
    pd.DataFrame(
        mask_results
    )
    .sort_values(
        "Correlation",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

display(
    mask_comparison_df
)

,Mask,Source,Mask voxels,Reference voxels,Correlation,MAE,RMSE
0,D0007_mean_b0_brain.nii_mask.nii.gz,NEW,1487914,1480549,0.922341,137164272.0,143790672.0
1,D0007_b0_brain.nii_mask.nii.gz,NEW,1435639,1429459,0.920190,139446288.0,145634624.0
2,D0007_eddy_mask.nii.gz,NEW,1470842,1461925,0.885821,135168864.0,142917760.0
3,D0007_mean_b0_brain_mask.nii.gz,NEW,1470842,1461925,0.885821,135168864.0,142917760.0


## 7. Best Legacy and Reconstructed Mask Candidates

The best legacy mask provides a benchmark for reproducing the original registration reference.

The best reconstructed candidate identifies the highest-performing mask that can be generated independently by the reconstructed pipeline.

In [18]:
old_results = (
    mask_comparison_df[
        mask_comparison_df[
            "Source"
        ] == "OLD"
    ]
)

new_results = (
    mask_comparison_df[
        mask_comparison_df[
            "Source"
        ] == "NEW"
    ]
)


print(
    "Best legacy/reference mask"
)
print(
    "=" * 70
)

if len(
    old_results
) > 0:

    display(
        old_results.head(1)
    )

else:

    print(
        "No compatible legacy mask found."
    )


print(
    "\nBest independently generated reconstructed mask"
)
print(
    "=" * 70
)

if len(
    new_results
) > 0:

    display(
        new_results.head(1)
    )

else:

    print(
        "No compatible reconstructed mask found."
    )

Best legacy/reference mask
No compatible legacy mask found.

Best independently generated reconstructed mask


,Mask,Source,Mask voxels,Reference voxels,Correlation,MAE,RMSE
0,D0007_mean_b0_brain.nii_mask.nii.gz,NEW,1487914,1480549,0.922341,137164272.0,143790672.0


## 8. Mask Coverage Comparison

Mask size alone does not determine registration quality, but large differences in spatial coverage can explain differences in the resulting registration reference.

The voxel count of each candidate mask is compared with the legacy mask benchmark.

In [19]:
if len(
    old_results
) > 0:

    benchmark_voxels = int(
        old_results.iloc[0][
            "Mask voxels"
        ]
    )

    coverage_comparison_df = (
        mask_comparison_df.copy()
    )

    coverage_comparison_df[
        "Voxel difference from legacy"
    ] = (
        coverage_comparison_df[
            "Mask voxels"
        ]
        - benchmark_voxels
    )

    coverage_comparison_df[
        "Relative voxel difference"
    ] = (
        coverage_comparison_df[
            "Voxel difference from legacy"
        ]
        / benchmark_voxels
    )

    display(
        coverage_comparison_df[
            [
                "Mask",
                "Source",
                "Mask voxels",
                "Voxel difference from legacy",
                "Relative voxel difference",
                "Correlation",
            ]
        ]
    )

else:

    print(
        "Legacy mask benchmark unavailable."
    )

Legacy mask benchmark unavailable.


## 9. Preliminary Mask Selection

The final mask should not be selected solely by similarity to the legacy mask.

The preferred reconstructed mask should:

1. be generated independently from reconstructed data,
2. provide reasonable brain coverage,
3. produce a strong registration reference,
4. support good downstream `bbregister` alignment,
5. preserve high final regional-profile reproducibility.

The best reconstructed candidate identified here will therefore be carried forward to registration and end-to-end validation.

In [20]:
if len(
    new_results
) > 0:

    best_new = (
        new_results.iloc[0]
    )

    preliminary_selection_df = pd.DataFrame([
        {
            "Selected mask": best_new[
                "Mask"
            ],
            "Correlation with legacy reference": best_new[
                "Correlation"
            ],
            "Mask voxels": int(
                best_new[
                    "Mask voxels"
                ]
            ),
            "Status": (
                "CANDIDATE FOR DOWNSTREAM VALIDATION"
            ),
        }
    ])

    display(
        preliminary_selection_df
    )

else:

    print(
        "No reconstructed candidate available for selection."
    )

,Selected mask,Correlation with legacy reference,Mask voxels,Status
0,D0007_mean_b0_brain.nii_mask.nii.gz,0.922341,1487914,CANDIDATE FOR DOWNSTREAM VALIDATION


# 10. BET Threshold Experiment

To evaluate whether the reconstructed registration mask can be improved independently, several FSL BET fractional intensity thresholds are tested using the same mean-b0 image.

Only the BET fractional intensity threshold (`-f`) is changed. All other inputs are held constant.

The tested thresholds are:

- 0.10
- 0.15
- 0.20
- 0.25
- 0.30

All generated masks are written only to the validation results directory and do not modify preprocessing outputs.

In [24]:
# ============================================================
# LOCATE MEAN B0 INPUT
# ============================================================

mean_b0_candidates = find_files(
    NEW_PIPELINE_ROOT,
    patterns=[
        f"{SUBJECT}_mean_b0.nii.gz",
    ],
    subject=SUBJECT,
)

if len(mean_b0_candidates) == 0:
    raise FileNotFoundError(
        f"No mean-b0 image found for {SUBJECT}"
    )

if len(mean_b0_candidates) > 1:
    print("Multiple exact mean-b0 candidates found:")
    for i, path in enumerate(mean_b0_candidates):
        print(f"[{i}] {path}")
    raise RuntimeError(
        "Expected one exact mean-b0 image."
    )

MEAN_B0 = mean_b0_candidates[0]

print("Mean b0 input:")
print(MEAN_B0)

Mean b0 input:
/mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007/diffusion/D0007_mean_b0.nii.gz


## 10.1 Load

In [26]:
BET_TEST_ROOT = (
    RESULTS_ROOT
    / "bet_threshold_test"
)

BET_THRESHOLDS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

BET_MASKS = {}

for threshold in BET_THRESHOLDS:

    threshold_label = (
        f"{threshold:.2f}"
        .replace(".", "p")
    )

    mask_path = (
        BET_TEST_ROOT
        / (
            f"{SUBJECT}_mean_b0_"
            f"bet_f{threshold_label}_mask.nii.gz"
        )
    )

    if not mask_path.exists():
        raise FileNotFoundError(
            f"Missing BET mask for f={threshold:.2f}:\n"
            f"{mask_path}"
        )

    BET_MASKS[threshold] = mask_path


print("Loaded BET masks:")

for threshold, path in BET_MASKS.items():
    print(
        f"f={threshold:.2f}: {path}"
    )

Loaded BET masks:
f=0.10: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/bet_threshold_test/D0007_mean_b0_bet_f0p10_mask.nii.gz
f=0.15: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/bet_threshold_test/D0007_mean_b0_bet_f0p15_mask.nii.gz
f=0.20: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/bet_threshold_test/D0007_mean_b0_bet_f0p20_mask.nii.gz
f=0.25: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/bet_threshold_test/D0007_mean_b0_bet_f0p25_mask.nii.gz
f=0.30: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/bet_threshold_test/D0007_mean_b0_bet_f0p30_mask.nii.gz


### 10.2 BET Mask Coverage

The spatial coverage of each BET-derived mask is quantified by its number of nonzero voxels.

Mask size is reported descriptively and is not used alone to determine the preferred threshold.

In [27]:
bet_coverage_results = []

for threshold, path in BET_MASKS.items():

    data = np.asarray(
        nib.load(path).dataobj,
        dtype=np.float32,
    )

    binary_mask = data > 0

    bet_coverage_results.append({
        "BET threshold": threshold,
        "Mask": path.name,
        "Mask voxels": int(
            binary_mask.sum()
        ),
    })


bet_coverage_df = (
    pd.DataFrame(
        bet_coverage_results
    )
    .sort_values(
        "BET threshold"
    )
    .reset_index(
        drop=True
    )
)

display(
    bet_coverage_df
)

,BET threshold,Mask,Mask voxels
0,0.10,D0007_mean_b0_bet_f0p10_mask.nii.gz,2899302
1,0.15,D0007_mean_b0_bet_f0p15_mask.nii.gz,1787849
2,0.20,D0007_mean_b0_bet_f0p20_mask.nii.gz,1514730
3,0.25,D0007_mean_b0_bet_f0p25_mask.nii.gz,1435411
4,0.30,D0007_mean_b0_bet_f0p30_mask.nii.gz,1376513


### 10.3 Registration Reference Comparison Across BET Thresholds

In [28]:
old_ref = np.asarray(
    nib.load(OLD_REG_REFERENCE).dataobj,
    dtype=np.float32,
)

summed_data = np.asarray(
    nib.load(SUMMED_DWI).dataobj,
    dtype=np.float32,
)

bet_reference_results = []

for threshold, mask_path in BET_MASKS.items():

    mask_data = np.asarray(
        nib.load(mask_path).dataobj,
        dtype=np.float32,
    )

    binary_mask = mask_data > 0

    candidate_ref = (
        summed_data
        * binary_mask
    )

    valid_mask = (
        np.isfinite(old_ref)
        & np.isfinite(candidate_ref)
        & (
            (old_ref != 0)
            | (candidate_ref != 0)
        )
    )

    old_values = old_ref[
        valid_mask
    ]

    new_values = candidate_ref[
        valid_mask
    ]

    difference = (
        new_values
        - old_values
    )

    bet_reference_results.append({
        "BET threshold": threshold,
        "Mask voxels": int(
            binary_mask.sum()
        ),
        "Reference voxels": int(
            np.count_nonzero(
                candidate_ref
            )
        ),
        "Correlation": np.corrcoef(
            old_values,
            new_values,
        )[0, 1],
        "MAE": np.mean(
            np.abs(difference)
        ),
        "RMSE": np.sqrt(
            np.mean(
                difference ** 2
            )
        ),
    })


bet_reference_df = (
    pd.DataFrame(
        bet_reference_results
    )
    .sort_values(
        "Correlation",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

display(
    bet_reference_df
)

,BET threshold,Mask voxels,Reference voxels,Correlation,MAE,RMSE
0,0.10,2899302,2862605,0.957915,98754864.0,111300176.0
1,0.15,1787849,1773116,0.943908,124681416.0,133900232.0
2,0.30,1376513,1371583,0.924569,142110096.0,147769120.0
3,0.20,1514730,1507097,0.924469,136081584.0,142892112.0
4,0.25,1435411,1429266,0.918750,139645856.0,145740000.0


In [29]:
# ============================================================
# LEGACY REGISTRATION SUPPORT
# ============================================================

legacy_support = (
    np.isfinite(old_ref)
    & (old_ref != 0)
)

legacy_voxels = int(
    legacy_support.sum()
)


overlap_results = []

for threshold, mask_path in BET_MASKS.items():

    mask_data = np.asarray(
        nib.load(mask_path).dataobj,
        dtype=np.float32,
    )

    bet_mask = (
        np.isfinite(mask_data)
        & (mask_data > 0)
    )

    shared = (
        legacy_support
        & bet_mask
    )

    shared_voxels = int(
        shared.sum()
    )

    bet_voxels = int(
        bet_mask.sum()
    )

    dice = (
        2 * shared_voxels
        / (
            legacy_voxels
            + bet_voxels
        )
    )

    legacy_recovered = (
        shared_voxels
        / legacy_voxels
    )

    bet_overlap_fraction = (
        shared_voxels
        / bet_voxels
    )

    extra_voxels = int(
        (
            bet_mask
            & ~legacy_support
        ).sum()
    )

    missed_voxels = int(
        (
            legacy_support
            & ~bet_mask
        ).sum()
    )

    overlap_results.append({
        "BET threshold": threshold,
        "Legacy voxels": legacy_voxels,
        "BET mask voxels": bet_voxels,
        "Shared voxels": shared_voxels,
        "Dice": dice,
        "Legacy recovered": legacy_recovered,
        "BET overlap with legacy": bet_overlap_fraction,
        "Extra voxels": extra_voxels,
        "Missed legacy voxels": missed_voxels,
    })


bet_overlap_df = (
    pd.DataFrame(overlap_results)
    .sort_values(
        "Dice",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(bet_overlap_df)

,BET threshold,Legacy voxels,BET mask voxels,Shared voxels,Dice,Legacy recovered,BET overlap with legacy,Extra voxels,Missed legacy voxels
0,0.30,1296848,1376513,1293768,0.967896,0.997625,0.939888,82745,3080
1,0.25,1296848,1435411,1294683,0.947702,0.998331,0.901960,140728,2165
2,0.20,1296848,1514730,1295275,0.921386,0.998787,0.855119,219455,1573
3,0.15,1296848,1787849,1295716,0.840093,0.999127,0.724735,492133,1132
4,0.10,1296848,2899302,1296284,0.617844,0.999565,0.447102,1603018,564


In [30]:
bet_evaluation_df = (
    bet_reference_df
    .merge(
        bet_overlap_df,
        on="BET threshold",
        how="inner",
        suffixes=(
            "_reference",
            "_overlap",
        ),
    )
)

columns_to_show = [
    "BET threshold",
    "Mask voxels",
    "Correlation",
    "Dice",
    "Legacy recovered",
    "BET overlap with legacy",
    "Extra voxels",
    "Missed legacy voxels",
]

display(
    bet_evaluation_df[
        columns_to_show
    ].sort_values(
        "BET threshold"
    )
)

,BET threshold,Mask voxels,Correlation,Dice,Legacy recovered,BET overlap with legacy,Extra voxels,Missed legacy voxels
0,0.10,2899302,0.957915,0.617844,0.999565,0.447102,1603018,564
1,0.15,1787849,0.943908,0.840093,0.999127,0.724735,492133,1132
3,0.20,1514730,0.924469,0.921386,0.998787,0.855119,219455,1573
4,0.25,1435411,0.918750,0.947702,0.998331,0.901960,140728,2165
2,0.30,1376513,0.924569,0.967896,0.997625,0.939888,82745,3080


## 11. Downstream Registration Comparison

The BET threshold experiment identified three plausible reconstructed masks for downstream registration:

- `f = 0.20`
- `f = 0.25`
- `f = 0.30`

The `f = 0.10` and `f = 0.15` masks were excluded because they introduced substantially greater spatial coverage outside the legacy registration support.

Each retained mask will now be used to construct a 3D registration reference and run `bbregister`.

The resulting transforms will be compared with the legacy `DWI2T1` transform to determine which independently generated BET mask provides the best downstream registration performance.

In [31]:
FINAL_BET_THRESHOLDS = [
    0.20,
    0.25,
    0.30,
]

FINAL_BET_MASKS = {
    threshold: BET_MASKS[threshold]
    for threshold in FINAL_BET_THRESHOLDS
}

print("BET masks retained for registration:")

for threshold, path in FINAL_BET_MASKS.items():
    print(
        f"f={threshold:.2f}: {path.name}"
    )

BET masks retained for registration:
f=0.20: D0007_mean_b0_bet_f0p20_mask.nii.gz
f=0.25: D0007_mean_b0_bet_f0p25_mask.nii.gz
f=0.30: D0007_mean_b0_bet_f0p30_mask.nii.gz


In [32]:
REG_TEST_ROOT = (
    RESULTS_ROOT
    / "registration_test"
)

REG_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REGISTRATION_REFERENCES = {}

for threshold, mask_path in FINAL_BET_MASKS.items():

    label = (
        f"{threshold:.2f}"
        .replace(".", "p")
    )

    output_path = (
        REG_TEST_ROOT
        / f"{SUBJECT}_regref_bet_f{label}.nii.gz"
    )

    mask_data = np.asarray(
        nib.load(mask_path).dataobj,
        dtype=np.float32,
    )

    candidate_ref = (
        summed_data
        * (mask_data > 0)
    )

    output_img = nib.Nifti1Image(
        candidate_ref.astype(np.float32),
        summed_img.affine,
        summed_img.header,
    )

    nib.save(
        output_img,
        output_path,
    )

    REGISTRATION_REFERENCES[
        threshold
    ] = output_path


print("Registration references created:")

for threshold, path in REGISTRATION_REFERENCES.items():
    print(
        f"f={threshold:.2f}: {path}"
    )

Registration references created:
f=0.20: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/registration_test/D0007_regref_bet_f0p20.nii.gz
f=0.25: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/registration_test/D0007_regref_bet_f0p25.nii.gz
f=0.30: /mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/validation/results/D0007/registration_mask_comparison/registration_test/D0007_regref_bet_f0p30.nii.gz


### 11.3 Registration Transform Comparison

The `bbregister` transforms generated from the retained BET masks are compared with the legacy `DWI2T1` transform.

For each candidate, the comparison reports:

- maximum absolute matrix-element difference,
- mean absolute matrix-element difference,
- translation differences along x, y, and z,
- overall translation-difference magnitude.

Smaller differences indicate closer reproduction of the legacy registration geometry.

In [33]:
# ============================================================
# LOAD CANDIDATE TRANSFORMS
# ============================================================

FINAL_BET_THRESHOLDS = [0.20, 0.25, 0.30]

REGISTRATION_TRANSFORMS = {}

for threshold in FINAL_BET_THRESHOLDS:

    label = f"{threshold:.2f}".replace(".", "p")

    path = (
        REG_TEST_ROOT
        / f"{SUBJECT}_DWI2T1_bet_f{label}.dat"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing transform for f={threshold:.2f}:\n{path}"
        )

    REGISTRATION_TRANSFORMS[threshold] = path


# ============================================================
# LOCATE LEGACY TRANSFORM
# ============================================================

old_transform_candidates = find_files(
    OLD_PIPELINE_ROOT,
    patterns=[
        "*DWI2T1*.dat",
        "*dwi2t1*.dat",
    ],
    subject=SUBJECT,
)

print("Legacy transform candidates:")
for i, path in enumerate(old_transform_candidates):
    print(f"[{i}] {path}")

if len(old_transform_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one legacy transform. "
        "Inspect the candidate list above."
    )

OLD_TRANSFORM = old_transform_candidates[0]

print("\nSelected legacy transform:")
print(OLD_TRANSFORM)


# ============================================================
# LOAD FREESURFER .DAT MATRIX
# ============================================================

def load_bbregister_matrix(path):
    """
    Extract the first 4x4 numeric matrix from a FreeSurfer .dat file.
    """
    lines = Path(path).read_text().splitlines()

    rows = []

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 4:
            continue

        try:
            row = [float(x) for x in parts]
        except ValueError:
            continue

        rows.append(row)

    if len(rows) < 4:
        raise ValueError(
            f"Could not locate a 4x4 matrix in {path}"
        )

    return np.asarray(
        rows[:4],
        dtype=float,
    )


old_matrix = load_bbregister_matrix(
    OLD_TRANSFORM
)


# ============================================================
# COMPARE TRANSFORMS
# ============================================================

results = []

for threshold, path in REGISTRATION_TRANSFORMS.items():

    new_matrix = load_bbregister_matrix(
        path
    )

    diff = new_matrix - old_matrix

    translation_diff = (
        new_matrix[:3, 3]
        - old_matrix[:3, 3]
    )

    results.append({
        "BET threshold": threshold,
        "Max abs matrix difference": np.max(
            np.abs(diff)
        ),
        "Mean abs matrix difference": np.mean(
            np.abs(diff)
        ),
        "Translation dx": translation_diff[0],
        "Translation dy": translation_diff[1],
        "Translation dz": translation_diff[2],
        "Translation difference magnitude": np.linalg.norm(
            translation_diff
        ),
    })


transform_comparison_df = (
    pd.DataFrame(results)
    .sort_values(
        "Translation difference magnitude"
    )
    .reset_index(drop=True)
)

display(
    transform_comparison_df
)


# ============================================================
# BEST CANDIDATE
# ============================================================

best_registration = (
    transform_comparison_df.iloc[0]
)

print("\nBest candidate based on transform similarity:")
print(
    f"BET threshold: "
    f"{best_registration['BET threshold']:.2f}"
)

print(
    f"Translation difference magnitude: "
    f"{best_registration['Translation difference magnitude']:.6f}"
)

print(
    f"Max matrix difference: "
    f"{best_registration['Max abs matrix difference']:.6f}"
)

Legacy transform candidates:
[0] /mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/DWI2T1_dti.dat

Selected legacy transform:
/mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/DWI2T1_dti.dat


,BET threshold,Max abs matrix difference,Mean abs matrix difference,Translation dx,Translation dy,Translation dz,Translation difference magnitude
0,0.25,0.055459,0.006375,-0.000094,0.055459,-0.040270,0.068538
1,0.30,0.056238,0.006387,-0.000134,0.056238,-0.039784,0.068887
2,0.20,0.057188,0.006752,-0.002221,0.057188,-0.042159,0.071082



Best candidate based on transform similarity:
BET threshold: 0.25
Translation difference magnitude: 0.068538
Max matrix difference: 0.055459


In [35]:
# ============================================================
# LOCATE LEGACY MASK / LEGACY REGISTRATION SUPPORT
# ============================================================
MASKROOT = RESULTS_ROOT / "bet_threshold_test"
legacy_mask_candidates = find_files(
    OLD_PIPELINE_ROOT,
    patterns=[
        f"{SUBJECT}_mask.nii.gz",
        f"{SUBJECT}_subjspace_mask.nii.gz",
        "*subjspace_mask*.nii.gz",
    ],
    subject=SUBJECT,
)

legacy_regref_candidates = find_files(
    OLD_PIPELINE_ROOT,
    patterns=[
        "*dwi_masked.nii.gz",
        "*sum_masked*.nii.gz",
        "*summed_masked*.nii.gz",
    ],
    subject=SUBJECT,
)

print("Legacy binary-mask candidates:")
for p in legacy_mask_candidates:
    print("  ", p)

print("\nLegacy masked-DWI candidates:")
for p in legacy_regref_candidates:
    print("  ", p)


# ============================================================
# SELECT LEGACY SUPPORT
# ============================================================

if len(legacy_mask_candidates) == 1:

    LEGACY_SOURCE = legacy_mask_candidates[0]

    legacy_img = nib.load(LEGACY_SOURCE)
    legacy_support = legacy_img.get_fdata() > 0

    legacy_source_type = "binary mask"

elif len(legacy_mask_candidates) == 0 and len(legacy_regref_candidates) == 1:

    LEGACY_SOURCE = legacy_regref_candidates[0]

    legacy_img = nib.load(LEGACY_SOURCE)
    legacy_support = legacy_img.get_fdata() != 0

    legacy_source_type = "nonzero support of masked DWI"

else:
    raise RuntimeError(
        "Could not uniquely determine the legacy mask/support.\n"
        "Inspect the candidate lists above."
    )

print("\nSelected legacy source:")
print(LEGACY_SOURCE)
print("Source type:", legacy_source_type)
print("Legacy support voxels:", int(legacy_support.sum()))


# ============================================================
# LOAD BET CANDIDATES
# ============================================================

BET_COMPARE = {
    0.25: (
        MASKROOT
        / f"{SUBJECT}_mean_b0_bet_f0p25_mask.nii.gz"
    ),
    0.30: (
        MASKROOT
        / f"{SUBJECT}_mean_b0_bet_f0p30_mask.nii.gz"
    ),
}


# ============================================================
# CHECK GEOMETRY + COMPARE
# ============================================================

rows = []

for threshold, mask_path in BET_COMPARE.items():

    mask_img = nib.load(mask_path)

    if mask_img.shape != legacy_img.shape:
        raise ValueError(
            f"Shape mismatch for f={threshold}:\n"
            f"legacy: {legacy_img.shape}\n"
            f"new:    {mask_img.shape}"
        )

    if not np.allclose(
        mask_img.affine,
        legacy_img.affine,
        atol=1e-4,
    ):
        raise ValueError(
            f"Affine mismatch for f={threshold}"
        )

    new_support = mask_img.get_fdata() > 0

    intersection = np.logical_and(
        legacy_support,
        new_support,
    ).sum()

    union = np.logical_or(
        legacy_support,
        new_support,
    ).sum()

    legacy_n = legacy_support.sum()
    new_n = new_support.sum()

    dice = (
        2 * intersection
        / (legacy_n + new_n)
    )

    jaccard = intersection / union

    extra = np.logical_and(
        new_support,
        ~legacy_support,
    ).sum()

    missed = np.logical_and(
        legacy_support,
        ~new_support,
    ).sum()

    legacy_recall = (
        intersection / legacy_n
    )

    rows.append({
        "BET threshold": threshold,
        "Legacy voxels": int(legacy_n),
        "New mask voxels": int(new_n),
        "Dice": dice,
        "Jaccard": jaccard,
        "Legacy coverage retained": legacy_recall,
        "Extra voxels": int(extra),
        "Missed legacy voxels": int(missed),
    })


legacy_mask_comparison_df = (
    pd.DataFrame(rows)
    .sort_values(
        "Dice",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(legacy_mask_comparison_df)


# ============================================================
# SUMMARY
# ============================================================

best = legacy_mask_comparison_df.iloc[0]

print(
    "\nMost similar candidate to legacy support:"
)

print(
    f"BET threshold = {best['BET threshold']:.2f}"
)

print(
    f"Dice = {best['Dice']:.4f}"
)

print(
    f"Legacy coverage retained = "
    f"{best['Legacy coverage retained']:.4%}"
)

print(
    f"Extra voxels = "
    f"{int(best['Extra voxels']):,}"
)

print(
    f"Missed legacy voxels = "
    f"{int(best['Missed legacy voxels']):,}"
)

Legacy binary-mask candidates:

Legacy masked-DWI candidates:
   /mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/D0007_dwi_masked.nii.gz

Selected legacy source:
/mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/D0007_dwi_masked.nii.gz
Source type: nonzero support of masked DWI
Legacy support voxels: 1296848


,BET threshold,Legacy voxels,New mask voxels,Dice,Jaccard,Legacy coverage retained,Extra voxels,Missed legacy voxels
0,0.30,1296848,1376513,0.967896,0.937790,0.997625,82745,3080
1,0.25,1296848,1435411,0.947702,0.900601,0.998331,140728,2165



Most similar candidate to legacy support:
BET threshold = 0.30
Dice = 0.9679
Legacy coverage retained = 99.7625%
Extra voxels = 82,745
Missed legacy voxels = 3,080


In [37]:
from pathlib import Path
import numpy as np
import pandas as pd

SUBJECT = "D0007"

WORK = Path("/mnt/newStor/paros/paros_WORK/xintong")

OLD_DIR = (
    WORK
    / "test"
    / "existing_pipeline"
    / "output"
    / SUBJECT
    / "adc"
    / "adc_cols_region_mean"
)

NEW_DIR = (
    WORK
    / "test"
    / "new_pipeline"
    / f"{SUBJECT}_bet030"
    / "regional_profiles"
    / "region_mean"
)

print("OLD_DIR:", OLD_DIR)
print("NEW_DIR:", NEW_DIR)

old_files = sorted(OLD_DIR.glob("*.csv"))
new_files = sorted(NEW_DIR.glob("*.csv"))

print("\nOLD files:", len(old_files))
print("NEW files:", len(new_files))

print("\nOLD examples:")
for p in old_files[:5]:
    print(" ", p.name)

print("\nNEW examples:")
for p in new_files[:5]:
    print(" ", p.name)


# ============================================================
# Match files by region name
# ============================================================

def normalize_name(path):
    name = path.stem.lower()

    for token in [
        "adc",
        "region",
        "mean",
        "profile",
        "cols",
    ]:
        name = name.replace(token, "")

    name = name.replace("__", "_")
    name = name.strip("_- ")

    return name


old_map = {
    normalize_name(p): p
    for p in old_files
}

new_map = {
    normalize_name(p): p
    for p in new_files
}

common_keys = sorted(
    set(old_map) & set(new_map)
)

print("\nMatched regions:", len(common_keys))

if len(common_keys) == 0:
    raise RuntimeError(
        "No matched regional profile files found."
    )


# ============================================================
# Load matched regional profiles
# ============================================================

old_values = []
new_values = []
region_rows = []

for key in common_keys:

    old_df = pd.read_csv(
        old_map[key],
        header=None,
    )

    new_df = pd.read_csv(
        new_map[key],
        header=None,
    )

    old_arr = (
        old_df
        .select_dtypes(include=[np.number])
        .to_numpy()
        .ravel()
    )

    new_arr = (
        new_df
        .select_dtypes(include=[np.number])
        .to_numpy()
        .ravel()
    )

    old_arr = old_arr[
        np.isfinite(old_arr)
    ]

    new_arr = new_arr[
        np.isfinite(new_arr)
    ]

    if old_arr.shape != new_arr.shape:
        print(
            f"Skipping {key}: "
            f"OLD {old_arr.shape} vs NEW {new_arr.shape}"
        )
        continue

    old_values.extend(
        old_arr.tolist()
    )

    new_values.extend(
        new_arr.tolist()
    )

    if len(old_arr) > 1:
        region_corr = np.corrcoef(
            old_arr,
            new_arr,
        )[0, 1]
    else:
        region_corr = np.nan

    region_rows.append({
        "Region": key,
        "N values": len(old_arr),
        "Correlation": region_corr,
        "MAE": np.mean(
            np.abs(new_arr - old_arr)
        ),
    })


old_values = np.asarray(
    old_values,
    dtype=float,
)

new_values = np.asarray(
    new_values,
    dtype=float,
)

print("\nTotal matched values:")
print("OLD:", old_values.shape)
print("NEW:", new_values.shape)


# ============================================================
# Overall metrics
# ============================================================

corr = np.corrcoef(
    old_values,
    new_values,
)[0, 1]

diff = new_values - old_values

mae = np.mean(
    np.abs(diff)
)

rmse = np.sqrt(
    np.mean(diff ** 2)
)

mean_diff = np.mean(
    diff
)

summary_df = pd.DataFrame({
    "Metric": [
        "Correlation",
        "MAE",
        "RMSE",
        "Mean difference",
    ],
    "Value": [
        corr,
        mae,
        rmse,
        mean_diff,
    ],
})

display(summary_df)

print("\nFinal BET f=0.30 validation:")
print(f"Correlation     = {corr:.6f}")
print(f"MAE             = {mae:.6f}")
print(f"RMSE            = {rmse:.6f}")
print(f"Mean difference = {mean_diff:.6f}")


# Optional region-level table
region_comparison_df = pd.DataFrame(
    region_rows
).sort_values(
    "MAE"
)

display(region_comparison_df)

OLD_DIR: /mnt/newStor/paros/paros_WORK/xintong/test/existing_pipeline/output/D0007/adc/adc_cols_region_mean
NEW_DIR: /mnt/newStor/paros/paros_WORK/xintong/test/new_pipeline/D0007_bet030/regional_profiles/region_mean

OLD files: 68
NEW files: 68

OLD examples:
  D0007_lh_bankssts_cols_adc_mean.csv
  D0007_lh_caudalanteriorcingulate_cols_adc_mean.csv
  D0007_lh_caudalmiddlefrontal_cols_adc_mean.csv
  D0007_lh_cuneus_cols_adc_mean.csv
  D0007_lh_entorhinal_cols_adc_mean.csv

NEW examples:
  D0007_lh_bankssts_adc_mean.csv
  D0007_lh_caudalanteriorcingulate_adc_mean.csv
  D0007_lh_caudalmiddlefrontal_adc_mean.csv
  D0007_lh_cuneus_adc_mean.csv
  D0007_lh_entorhinal_adc_mean.csv

Matched regions: 68

Total matched values:
OLD: (1428,)
NEW: (1428,)


,Metric,Value
0,Correlation,0.995826
1,MAE,0.000013
2,RMSE,0.000021
3,Mean difference,0.000006



Final BET f=0.30 validation:
Correlation     = 0.995826
MAE             = 0.000013
RMSE            = 0.000021
Mean difference = 0.000006


,Region,N values,Correlation,MAE
7,d0007_lh_inferiorparietal,21,0.999997,0.000001
61,d0007_rh_rostralmiddlefrontal,21,0.999959,0.000001
39,d0007_rh_frontalpole,21,0.999662,0.000003
5,d0007_lh_frontalpole,21,0.999892,0.000003
18,d0007_lh_parsopercularis,21,0.999958,0.000003
...,...,...,...,...
14,d0007_lh_medialorbitofrontal,21,0.999853,0.000025
55,d0007_rh_pericalcarine,21,0.999327,0.000039
45,d0007_rh_lateraloccipital,21,0.999418,0.000055
38,d0007_rh_entorhinal,21,0.997535,0.000057
